# MLflow y selección de modelo

Este notebook consulta los runs ya existentes del experimento `invoice-risk`. No entrena modelos, no crea runs, no carga artifacts y no promueve nada al Model Registry. Su propósito es preparar evidencia para una decisión humana.

In [1]:
import os

import mlflow

tracking_uri = os.environ.get('MLFLOW_TRACKING_URI')
if not tracking_uri:
    raise RuntimeError('Configure MLFLOW_TRACKING_URI before running this notebook.')

mlflow.set_tracking_uri(tracking_uri)
experiment = mlflow.get_experiment_by_name('invoice-risk')
if experiment is None:
    raise RuntimeError(
        'The invoice-risk experiment was not found at MLFLOW_TRACKING_URI. '
        'Start the tracking server and prepare its runs before running this notebook.'
    )

print(f'Consulting invoice-risk at {tracking_uri}')

Consulting invoice-risk at sqlite:////Users/admin/Desktop/elements/usach/diplomado/invoice_ops/invoice-ai/var/mlflow.db


## Comparación inicial

La tabla conserva solo los identificadores, métricas y metadatos necesarios para comparar runs reproducibles. El orden inicial es por `recall` descendente porque dejar pasar una factura que requería revisión es un falso negativo; no obstante, ordenar no equivale a elegir automáticamente un modelo.

In [2]:
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
if runs.empty:
    raise RuntimeError('The invoice-risk experiment has no prepared runs to compare.')

comparison = (
    runs.loc[:, [
        'run_id',
        'params.model_type',
        'metrics.accuracy',
        'metrics.precision',
        'metrics.recall',
        'metrics.f1',
        'metrics.roc_auc',
        'params.dataset_version',
        'tags.git_commit',
    ]]
    .rename(columns={
        'params.model_type': 'model_type',
        'metrics.accuracy': 'accuracy',
        'metrics.precision': 'precision',
        'metrics.recall': 'recall',
        'metrics.f1': 'f1',
        'metrics.roc_auc': 'roc_auc',
        'params.dataset_version': 'dataset_version',
        'tags.git_commit': 'git_commit',
    })
    .sort_values('recall', ascending=False, na_position='last')
    .reset_index(drop=True)
)

display(comparison)

,run_id,model_type,accuracy,precision,recall,f1,roc_auc,dataset_version,git_commit
0,24619abbb7b24edb97247af78213dc52,random_forest,0.788333,0.486111,0.185676,0.268714,0.643503,invoice-risk-v1,cacc0442cff9114b9cc550c993133163fca48fae
1,409c248f309444299e9a4841f2745458,random_forest,0.788333,0.486111,0.185676,0.268714,0.643503,invoice-risk-v1,cacc0442cff9114b9cc550c993133163fca48fae
2,a8d23cec83884b3987b90a1440ff1c3e,logistic,0.800000,0.634921,0.106101,0.181818,0.667080,invoice-risk-v1,cacc0442cff9114b9cc550c993133163fca48fae
3,4390efc242524c8493c6e45effda5b6f,logistic,0.800000,0.634921,0.106101,0.181818,0.667080,invoice-risk-v1,cacc0442cff9114b9cc550c993133163fca48fae
4,211a7fada22b4fa59f4a28f92fb7b068,dummy,0.790556,0.000000,0.000000,0.000000,0.500000,invoice-risk-v1,cacc0442cff9114b9cc550c993133163fca48fae
5,df637af27ca74f3292590d04b90ec8fd,dummy,0.790556,0.000000,0.000000,0.000000,0.500000,invoice-risk-v1,cacc0442cff9114b9cc550c993133163fca48fae


## La decisión sigue siendo humana

El primer lugar por `recall` es un candidato para discusión, no una selección automática. Antes de elegir, el equipo responsable debe responder:

1. ¿Qué modelo es el mejor candidato para este caso de uso?
2. ¿Qué criterio de selección debe prevalecer: recall, precision, F1, ROC AUC, accuracy o una combinación explícita?
3. ¿Cuál es el coste operativo y de riesgo de los falsos negativos: facturas que requerían revisión y el modelo dejó pasar?
4. ¿Qué trade-offs entre más detecciones, falsos positivos y capacidad de revisión manual acepta el negocio?

Un mayor recall puede reducir falsos negativos, pero normalmente aumenta revisiones manuales y potencialmente falsos positivos. La tabla ordenada muestra evidencia; no define la política ni autoriza una promoción.

In [3]:
candidate = comparison.iloc[0]
print(
    'Candidate for human review: '
    f"{candidate['model_type']} (run_id={candidate['run_id']}, recall={candidate['recall']:.3f})"
)

Candidate for human review: random_forest (run_id=24619abbb7b24edb97247af78213dc52, recall=0.186)


## Tracking no es Registry

MLflow Tracking registra experimentos, runs, parámetros, métricas y metadatos para compararlos. MLflow Model Registry administra el ciclo de vida de versiones de modelos que ya fueron aprobadas, por ejemplo mediante registro, aliases o etapas según la política del equipo. Este notebook solo consulta Tracking: no carga modelos o artifacts y no registra, promueve ni cambia el estado de ningún modelo.